**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2021**		

**Лабораторная работа №4. Критерии принятия решения и метрики качества**

**Цель работы:** изучение основных критериев принятия решения и метрик качества, рассматриваемых при решении задачи голосовой биометрии.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать классические критерии принятия решения (критерий максимального правдоподобия, критерий максимума апостериорной вероятности, критерий Неймана-Пирсона, критерий Байеса и минимаксный критерий), позволяющие автоматически подобрать порог принятия решения для биометрической системы, а также оценить качество полученных биометрических решений по целевым метрикам.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать тестовую часть базы [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка тестовых данных и протокола для проведения тестирования системы верификации диктора.							

2. Вычисление дикторских моделей для эталонных и тестовых звукозаписей используемого протокола.

3. Формирование оценок сравнения эталонных и тестовых дикторских моделей.

4. Построение таргет- и импостор-гистограмм оценок сравнения моделей.

5. Вычисление порогового значения, вероятности ошибки принятия решения, вероятностей ошибок первого и второго рода: критерий максимального правдоподобия, критерий максимума апостериорной вероятности (критерий минимума вероятности ошибки), критерий Неймана-Пирсона, критерий Байеса и минимаксный критерий.

6. Вычисление значения равновероятной ошибки (EER, equal error rate).							

In [ ]:
# Автоматически перезагружаем изменённые модули перед выполнением кода
%load_ext autoreload
%autoreload 2

# Импортируем необходимые модули
import itertools
import os
import sys
from math import sqrt
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import numpy as np
import torch
from matplotlib.pyplot import grid, hist, legend, plot, show, title, xlabel, ylabel
from torch.utils.data import DataLoader

from common import compute_scores, download_dataset, download_protocol, extract_dataset
from common import extract_features, test_dataset_loader
from constants import (
    CRITERION_BAYES,
    CRITERION_MAP,
    CRITERION_MINIMAX,
    CRITERION_NEYMAN_PEARSON,
    DATASET_LIST_PATH,
    DATA_DIR,
    DEFAULT_COST_CORRECT_IMPOSTOR,
    DEFAULT_COST_CORRECT_TARGET,
    DEFAULT_COST_FALSE_ACCEPT,
    DEFAULT_COST_MISSED_TARGET,
    DEFAULT_MODEL_DIR,
    DEFAULT_NEYMAN_PEARSON_FNR,
    DEFAULT_PROTOCOL_PATH,
    DEFAULT_TARGET_PRIOR,
    DEFAULT_VOXCELEB_DIR,
    EER_RESULT_MESSAGE,
    EMBEDDING_SIZE,
    ENCODER_ATTENTIVE_STATISTICS_POOLING,
    ERROR_VOXCELEB_CREDENTIALS,
    EVALUATION_FRAMES,
    EVALUATION_SEGMENTS,
    EVALUATION_WORKERS,
    LABEL_DENSITY,
    LABEL_ERROR_PROBABILITY,
    LABEL_IMPOSTOR_DENSITY,
    LABEL_IMPOSTOR_SCORES,
    LABEL_LLR,
    LABEL_SCORE,
    LABEL_TARGET_DENSITY,
    LABEL_TARGET_SCORES,
    LABEL_THRESHOLD,
    MEL_FILTERS,
    MINIMAX_PRIOR_STEPS,
    MINIMAX_RESULT_MESSAGE,
    MODEL_LIST_PATH,
    NEYMANN_PEARSON_RESULT_MESSAGE,
    PROTOCOL_LIST_PATH,
    RESULT_MESSAGE,
    TEXT_ENCODING,
    TITLE_ERROR_PROBABILITY,
    TITLE_LLR,
    TITLE_SCORE_DISTRIBUTIONS,
    VOXCELEB_ARCHIVE_PATH,
    VOXCELEB_EXTRACT_PATH,
    VOXCELEB_PASSWORD_ENV,
    VOXCELEB_USER_ENV,
)
from exercises_blank import (
    bayes_test,
    decision_curve,
    equal_error_rate,
    gaussian_pdf,
    llr,
    map_test,
    minmax_test,
    neyman_pearson_test,
    tar_imp_hists,
)
from LoadModel import load_model
from ResNetSE34V2 import MainModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

**1. Подготовка тестовых данных и протокола для проведения тестирования системы верификации диктора**

В ходе выполнения лабораторной работы необходимы *тестовые данные*, а также связанный с ними *протокол тестирования*. Данные представляют собой звукозаписи голосов людей мужского и женского пола, сохраненных в формат *wav*. Протокол тестирования организован в виде текстового файла, в каждой строчке которого содержится одна таргет- или импостор-попытка сравнения эталонной и тестовой звукозаписи (рассматривается задача *верификации диктора*, т.е. сравнение «один к одному»). Голоса эталона и теста из *таргет-попытки* (метка «1») соответствуют одному и тому же диктору, а голоса эталона и теста из *импостор-попытки* (метка «0») соответствуют разным дикторам.

Необходимые данные можно скачать по ссылкам, представленным ниже:

1. Выполнить загрузку и распаковку звуковых wav-файлов базы [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip).

2. Выполнить загрузку протокола тестирования [VoxCeleb1-O cleaned](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt).

![Рисунок 1](../images/voxceleb_dataset.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [ ]:
# Загружаем тестовую часть VoxCeleb1. Данные доступа не храним в ноутбуке.
voxceleb_user = os.environ.get(VOXCELEB_USER_ENV)
voxceleb_password = os.environ.get(VOXCELEB_PASSWORD_ENV)
if not voxceleb_user or not voxceleb_password:
    raise RuntimeError(ERROR_VOXCELEB_CREDENTIALS)

with DATASET_LIST_PATH.open('r', encoding=TEXT_ENCODING) as dataset_list_file:
    dataset_lines = dataset_list_file.readlines()

download_dataset(
    dataset_lines,
    user=voxceleb_user,
    password=voxceleb_password,
    save_path=str(DATA_DIR),
)

In [ ]:
# Распаковываем тестовую часть VoxCeleb1
extract_dataset(
    save_path=str(VOXCELEB_EXTRACT_PATH),
    fname=str(VOXCELEB_ARCHIVE_PATH),
)

In [ ]:
# Загружаем очищенный протокол верификации VoxCeleb1-O
with PROTOCOL_LIST_PATH.open('r', encoding=TEXT_ENCODING) as protocol_list_file:
    protocol_lines = protocol_list_file.readlines()

download_protocol(protocol_lines, save_path=str(VOXCELEB_EXTRACT_PATH))

**2. Вычисление дикторских моделей для эталонных и тестовых звукозаписей используемого протокола**

Эталонные и тестовые звукозаписи таргет- и импостор-попыток из тестового протокола неудобно сравнивать «в лоб». Рассматриваемые в лабораторной работе данные содержат звукозаписи с *частотой дискретизации* 16кГц. Последнее означает, что в 1 секунде записи содержится 16,000 отсчётов *цифрового сигнала*, а в двух секундах – 32,000 и т.п. Сравнивать между собой длинные вектора не самая удачная идея, т.к. это затратно в смысле вычислительной сложности. Дополнительно необходимо отметить, что «сырой» звуковой сигнал содержит не только голос диктора, а помехи и шумы, акустику помещения и т.п. Поэтому на практике при решении задачи распознавания диктора для эталонной и тестовой звукозаписи вычисляется *дикторская модель*, содержащая внутри себя основные характеристик голоса конкретного диктора. Построить дикторскую модель можно различными способами, например, с использованием специально обученной искусственной нейронной сети, на вход которой передаётся набор акустических признаков, вычисленных для определенной звукозаписи, а на выходе формируется вектор малой размерности, например, длины 128, 256 и т.п. Этот вектор, иногда называемый *эмбеддингом*, и является дикторской моделью.

Ниже представлена последовательность действий, которую необходимо выполнить при рассмотрении настоящего пункта:

1. Описать и инициализировать модель для генерации дикторских эмбеддингов.

2. Вгрузить в инициализированную модель обученные веса.

3. Вычислить эмбеддинги для протокольных эталонных и тестовых звукозаписей.

In [ ]:
# Инициализируем генератор дикторских эмбеддингов
model = MainModel(
    nOut=EMBEDDING_SIZE,
    encoder_type=ENCODER_ATTENTIVE_STATISTICS_POOLING,
    n_mels=MEL_FILTERS,
).to(device)

In [ ]:
# Загружаем предобученные параметры модели
with MODEL_LIST_PATH.open('r', encoding=TEXT_ENCODING) as model_list_file:
    model_urls = model_list_file.readlines()

model = load_model(model, model_urls, save_path=DEFAULT_MODEL_DIR)
model.eval()

In [ ]:
# Загружаем протокол тестирования
with DEFAULT_PROTOCOL_PATH.open('r', encoding=TEXT_ENCODING) as protocol_file:
    protocol_lines = protocol_file.readlines()

# Собираем уникальные имена аудиофайлов из всех сравнений
files = sorted(
    set(
        itertools.chain.from_iterable(
            line.strip().split()[-2:] for line in protocol_lines
        )
    )
)

# Создаём загрузчик и вычисляем эмбеддинги каждой записи один раз
test_dataset = test_dataset_loader(
    files,
    test_path=str(DEFAULT_VOXCELEB_DIR),
    eval_frames=EVALUATION_FRAMES,
    num_eval=EVALUATION_SEGMENTS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=EVALUATION_WORKERS,
)
features = extract_features(model, test_loader)

**3. Формирование оценок сравнения эталонных и тестовых дикторских моделей**

Для того, чтобы автоматически принять решении о принадлежности пары сравнения «эталон-тест» к таргет- или импостор-попытке, необходимо сформировать *оценку сравнения* между эталонной и тестовой дикторскими моделями. В простейшем случае подобные сравнения можно выполнить с использованием *евклидовой метрики*, *косинусной метрики* и т.п.

В рамках настоящего пункта необходимо выполнить следующее:

1. Выполнить формирование оценок сравнения эталонных и дикторских моделей с использованием любой подходящей метрики (евклидово расстояние, косинусное расстояние и т.п.).

2. Сформировать списки оценок сравнения, таргет-/импостор-меток и имен эталон-тест в триальных сравнениях.

In [ ]:
# Сравниваем эмбеддинги для каждой пары из протокола
all_scores, all_labels, all_trials = compute_scores(features, protocol_lines)

**4. Построение таргет- и импостор-гистограмм оценок сравнения моделей**

После вычисления оценок сравнения для таргет- и импостор-попыток из протокола тестирования можно провести первоначальную оценку качества работы системы распознавания диктора путем виузального анализа *гистограмм распределения таргет-* и *импостор-оценок*. Сила наложения гистограмм друг на друга определяет это качества. Чем сильнее друг на друга наложены гистограммы, тем качество работы системы распознавания диктора ниже и наоборот, чем меньше друг на друга наложены гистограммы, тем качество работы системы распознавания диктора выше. 

В рамках настоящего пункта необходимо выполнить следующее:

1. Разделить оценки сравнения на два набор: таргет-сравнения и импостор-сравнения.

2. Построить гистограммы для таргет-оценок и импостор-оценок.

In [ ]:
# Разделяем оценки на сравнения одного и разных дикторов
target_scores, impostor_scores = tar_imp_hists(all_scores, all_labels)

# Строим нормированные гистограммы оценок
minimum_score = np.concatenate((target_scores, impostor_scores)).min()
maximum_score = np.concatenate((target_scores, impostor_scores)).max()
hist(
    target_scores,
    int(sqrt(len(target_scores))),
    histtype='step',
    density=True,
    range=(minimum_score, maximum_score),
    label=LABEL_TARGET_SCORES,
)
hist(
    impostor_scores,
    int(sqrt(len(impostor_scores))),
    histtype='step',
    density=True,
    range=(minimum_score, maximum_score),
    label=LABEL_IMPOSTOR_SCORES,
)
xlabel(LABEL_SCORE)
ylabel(LABEL_DENSITY)
title(TITLE_SCORE_DISTRIBUTIONS)
grid()
legend()
show()

**5. Вычисление порогового значения, вероятности ошибки принятия решения, вероятностей ошибок первого и второго рода**

Выражаясь терминами *бинарной теории принятия решений* предположение о наличии таргет-пары при решении задачи распознавания диктора связывается с *нулевой гипотезой* – $H_0$, а предположение о наличии импостор-пары связывается с *альтернативной гипотезой* – $H_1$. Когда задача распознавания диктора рассматривается в качестве задачи верификации в системе возможно возникновение следующих ситуаций:

1. Пара сравнения «эталон-тест» является таргет-парой, решение системы принимается в пользу таргет-пары – событие $(D_0, H_0)$. Подобную ситуацию называют *правильным положительным решением (true positive)*.

2. Пара сравнения «эталон-тест» является таргет-парой, решение системы принимается в пользу импостор-пары – событие $(D_1, H_0)$. Подобную ситуацию называют *ложным отрицательным решением (false negative)*. Говорят, что в этом случае возникает *ошибка первого рода*.

3. Пара сравнения «эталон-тест» является импостор-парой, решение системы принимается в пользу таргет-пары – событие $(D_0, H_1)$. Подобную ситуацию называют *ложным положительным решением (false positive)*. Говорят, что в этом случае возникает *ошибка второго рода*.

4. Пара сравнения «эталон-тест» является импостор-парой, решение системы принимается в пользу импостор-пары – событие $(D_1, H_1)$. Подобную ситуацию называют *правильным отрицательным решением (true negative)*.

Как правило, перед принятием решения значение оценки сравнения эталона и теста представляется в виде *натурального логарифма отношения правдоподобия (LLR, log-likelihood ratio)*:

$$LLR = ln \left [ \frac{W(s|H_0)}{W(s|H_1)} \right ],$$ 

здесь $s$ – величина оценки сравнения эталона и теста, а $W(s|H_0)$ и $W(s|H_1)$ – плотности вероятностей, описывающие распределение оценок сравнения для таргет- и импостор-попыток. Обычно подобные плотности вероятностей рассматриваются в виде *одномерных гауссовских плотностей*, параметры математического ожидания и среднеквадратического отклонения которых оцениваются с использованием некоторого набора данных с помощью следующих математических формул:

$$m_{H_i} = \frac{1}{N_{H_i}}\sum_{n = 1}^{N_{H_i}}s_n, \sigma_{H_i} = \sqrt{\frac{1}{N_{H_i} - 1}\sum_{n = 1}^{N_{H_i}}(s_n - m_{H_i})^{2}}, i = 0, 1,$$

здесь $N_i$ – количество таргет-/импостор-пар в протоколе верификации, используемом для оценки параметров плотностей веротяностей $W(s|H_0)$ и $W(s|H_1)$. *В простейшем случае этот протокол может совпадать с тестовым протоколом, но ожидания по качеству работы системы распознавания диктора в этом случае будут завышенными!* Поэтому на практике протокол верификации для оценки параметров распределений выше выбирается отличным от протокола тестирования.

Для принятия автоматического решения биометрической системой требуется подобрать *величину порога принятия решения*. В случае, если значение $LLR$ для сравниваемой пары «эталон-тест» будет выше порога, решение биометрической системы будет принято в пользу таргет-пары и наоборот, если значение $LLR$ для сравниваемой пары «эталон-тест» будет ниже порога, решение биометрической системы будет принято в пользу импостор-пары.

Бинарная теория принятия решений говорит о том, что порог принятия решения может быть выбран несколькими различными способами: критерий максимального правдоподобия, критерий максимума апостериорной вероятности (критерий минимума вероятности ошибки), критерий Неймана-Пирсона, критерий Байеса и минимаксный критерий.

*Критерий максимального правдоподобия* предполагает, что априорные вероятности $P(H_0)$ и $P(H_1)$ выпадения таргет- и импостор-пар являются одинаковыми. Математически критерий может быть сформулирован следующим образом: если $LLR > 0$, то решение принимается в пользу таргет-пары и наоборот, если $LLR < 0$, то решение принимается в пользу импостор-пары. Величина порога принятия решения в этом случае равна 0.

*Критерий максимума апостериорной вероятности* является более общим, чем критерий максимального правдоподобия и учитывает возможность неравенства априорных вероятностей выпадения таргет- и импостор-пар. Математически критерий может быть сформулирован следующим образом: если $LLR > ln(P(H_1)/P(H_0))$, то решение принимается в пользу таргет-пары и наоборот, если $LLR < ln(P(H_1)/P(H_0))$, то решение принимается в пользу импостор-пары. Величина порога принятия решения в этом случае равна $ln(P(H_1)/P(H_0))$. Иногда рассматриваемый критерий называют *критерием минимума вероятности ошибки*, поскольку можно математически доказать, что рассматриваемый критерий следует из минимизации выражения для *вероятности ошибки системы принятия решения*: 

$$P_e = P(D_1, H_0) + P(D_0, H_1) = P(D_1|H_0)P(H_0) + P(D_0|H_1)P(H_1),$$

где $P(D_1, H_0)$ – вероятность события $(D_1, H_0)$, а $P(D_0, H_1)$ – вероятность события $(D_0, H_1)$. Вероятности $P(D_1|H_0)$ и $P(D_0|H_1)$ называют *вероятностями ошибок первого (FRR, false rejection rate)* и *второго рода (FAR, false acceptance rate)*.

*Критерий Неймана-Пирсона* предполагает, что одна из вероятностей ошибок, например, первого рода, строго задана. Величина порога принятия решения выбирается в точке, для которой вероятность другой ошибки, например, второго рода, будет минимальна. 

*Критерий Байеса* обобщает критерии, описанные выше. В его основе лежит минимизация *средней стоимости* или *байесовского риска*, который математически описывается следующим образом:

$$\overline{C} = C_{00}P(D_0|H_0)P(H_0) + C_{10}P(D_1|H_0)P(H_0) + C_{01}P(D_0|H_1)P(H_1) + C_{11}P(D_1|H_1)P(H_1),$$

где $C_{00}$, $C_{10}$, $C_{01}$ и $C_{11}$ представляют скалярные величины, описывающие стоимости, связанные с событиями $(D_0, H_0)$, $(D_1, H_0)$, $(D_0, H_1)$ и $(D_1, H_1)$ соответственно. Минимизация средней стоимости позволяет сформулировать критерий Байеса следующим образом: если $LLR > ln((C_{01} - C_{11})P(H_1)/[(C_{10} - C_{00})P(H_0)])$, то решение принимается в пользу таргет-пары и наоборот, если $LLR < ln((C_{01} - C_{11})P(H_1)/[(C_{10} - C_{00})P(H_0)])$, то решение принимается в пользу импостор-пары. Величина порога принятия решения в этом случае равна $ln((C_{01} - C_{11})P(H_1)/[(C_{10} - C_{00})P(H_0)])$.

При рассмотрении практических задач возможна ситуация, при которой априорные вероятности выпадения таргет- и импостор-пар являются неизвестными. В этом случае выбор порога принятия решения может быть выполнен с использованием *минимаксного критерия*. Выражение для среднего риска максимизируется по $P(H_0)$, а затем для найденного значения $P(H_0)$ минимизируется так, как в критерии Байеса.

В рамках настоящего пункта требуется выполнить следующие задания:

1. Вычислить логарифм отношение правдоподобия для всех оценок сравнения «эталон-тест» из рассматриваемого протокола тестирования. При решении данной задачи необходимо выполнить сортировку значений, в порядке возрастания, внутри одномерного массива, содержащего все оценки сравнения эталонов и тестов. Дополнительно рекомендуется синхронно с массивом выше сформировать одномерный массив, содержащий идеальную разметку сравнений эталонов и тестов.

2. Вычислить пороговое значение, верость ошибки принятия решения, вероятности ошибок первого и второга рода на основе оценок сравнения моделей с использованием критерия максимального правдоподобия и критерия максимума апостериорной вероятности (критерий минимума вероятности ошибки).

3. Вычислить пороговое значение, верость ошибки принятия решения, вероятность ошибки второга рода для заданной ошибки первого рода на основе оценок сравнения моделей с использованием критерия Неймана-Пирсона.

4. Вычислить пороговое значение, верость ошибки принятия решения, вероятности ошибок первого и второга рода на основе оценок сравнения моделей с использованием критерия Байеса.

5. Вычислить пороговое значение, верость ошибки принятия решения, вероятности ошибок первого и второга рода, минимум максимума байесовского риска, а также вероятность таргет-пары на основе оценок сравнения моделей с использованием минимаксного критерий.

In [ ]:
# Используем общую функцию плотности нормального распределения
gauss_pdf = gaussian_pdf

In [ ]:
# Вычисляем LLR для всех оценок сравнения
(
    sorted_ground_truth,
    sorted_scores,
    target_density,
    impostor_density,
    log_likelihood_ratio,
) = llr(
    all_scores,
    all_labels,
    target_scores,
    impostor_scores,
    gauss_pdf,
)

# Показываем гауссовские приближения распределений
plot(sorted_scores, target_density, label=LABEL_TARGET_DENSITY)
plot(sorted_scores, impostor_density, label=LABEL_IMPOSTOR_DENSITY)
xlabel(LABEL_SCORE)
ylabel(LABEL_DENSITY)
title(TITLE_LLR)
grid()
legend()
show()

In [ ]:
# Критерий максимума апостериорной вероятности
map_result = map_test(
    sorted_ground_truth,
    log_likelihood_ratio,
    target_scores,
    impostor_scores,
    DEFAULT_TARGET_PRIOR,
)
print(
    RESULT_MESSAGE.format(
        criterion=CRITERION_MAP,
        threshold=map_result[0],
        fnr=map_result[1],
        fpr=map_result[2],
        risk=map_result[3],
    )
)

# Показываем вероятность ошибки для каждого порога
thresholds, fnr_values, fpr_values = decision_curve(
    sorted_ground_truth,
    log_likelihood_ratio,
)
error_probabilities = (
    fnr_values * DEFAULT_TARGET_PRIOR
    + fpr_values * (1.0 - DEFAULT_TARGET_PRIOR)
)
plot(thresholds, error_probabilities)
xlabel(LABEL_THRESHOLD)
ylabel(LABEL_ERROR_PROBABILITY)
title(TITLE_ERROR_PROBABILITY)
grid()
show()

In [ ]:
# Критерий Неймана–Пирсона при заданном ограничении FNR
neyman_pearson_result = neyman_pearson_test(
    sorted_ground_truth,
    log_likelihood_ratio,
    target_scores,
    impostor_scores,
    DEFAULT_NEYMAN_PEARSON_FNR,
)
print(
    NEYMANN_PEARSON_RESULT_MESSAGE.format(
        criterion=CRITERION_NEYMAN_PEARSON,
        threshold=neyman_pearson_result[0],
        fnr=DEFAULT_NEYMAN_PEARSON_FNR,
        fpr=neyman_pearson_result[1],
    )
)

In [ ]:
# Критерий Байеса с одинаковыми стоимостями ошибок
bayes_result = bayes_test(
    sorted_ground_truth,
    log_likelihood_ratio,
    target_scores,
    impostor_scores,
    DEFAULT_TARGET_PRIOR,
    DEFAULT_COST_CORRECT_TARGET,
    DEFAULT_COST_MISSED_TARGET,
    DEFAULT_COST_FALSE_ACCEPT,
    DEFAULT_COST_CORRECT_IMPOSTOR,
)
print(
    RESULT_MESSAGE.format(
        criterion=CRITERION_BAYES,
        threshold=bayes_result[0],
        fnr=bayes_result[1],
        fpr=bayes_result[2],
        risk=bayes_result[3],
    )
)

In [ ]:
# Минимаксный критерий при неизвестной априорной вероятности
target_prior_grid = np.linspace(0.0, 1.0, MINIMAX_PRIOR_STEPS)
minimax_result = minmax_test(
    sorted_ground_truth,
    log_likelihood_ratio,
    target_scores,
    impostor_scores,
    target_prior_grid,
    DEFAULT_COST_CORRECT_TARGET,
    DEFAULT_COST_MISSED_TARGET,
    DEFAULT_COST_FALSE_ACCEPT,
    DEFAULT_COST_CORRECT_IMPOSTOR,
)
print(
    MINIMAX_RESULT_MESSAGE.format(
        criterion=CRITERION_MINIMAX,
        threshold=minimax_result[0],
        fnr=minimax_result[1],
        fpr=minimax_result[2],
        risk=minimax_result[3],
        prior=minimax_result[4],
    )
)

**6. Вычисление значения равновероятной ошибки**

Оценивать качество работы системы распознавания диктора визуально по наложению гистограмм для таргет- и импостор-оценок сравнения эталона и теста не очень удобно. Особенно тогда, когда количество сравниваемых системы очень большое. Поэтому можно воспользоваться некоторыми численными метриками автоматической оценки качества, например, вероятностью ошибки, как это было выполнено выше. В предположении, что априорные вероятности выпадения таргет- и импостор-пар сравнения эталонов и тестов являются одинаковыми, а также считая, что порог принятия решения выбран в точке, для которой вероятности ошибок первого и второго рода одинаковые, вероятность ошибки превращается *метрику равновероятной ошибки (EER, equal error rate)*.

Рассмотрим вычисление величины равновероятной ошибки для используемого протокола тестирования VoxCeleb1-O (cleaned).

In [ ]:
# Вычисляем EER по исходным оценкам сходства
eer, eer_threshold = equal_error_rate(
    np.asarray(all_labels, dtype=bool),
    np.asarray(all_scores, dtype=np.float64),
)
print(EER_RESULT_MESSAGE.format(eer=eer))

**7. Контрольные вопросы**
											
1. Как устроен протокол тестирования системы верификации?											
2. Что такое таргет- и импостор-пара?											
3. Какие подходы к сравнению дикторских моделей вам известны?
4. Что такое критерий принятия решения и какими они бывают (максимального правдоподобия, максимума апостериорной вероятности, Неймана-Пирсона, байесовский, минимаксный)?		
5. Что такое ошибки первого и второго рода и как вычислить их вероятности (FRR, false rejection rate, и FAR, false acceptance rate) в распознавании диктора?
6. Что такое равный уровень ошибок (EER, equal error rate)?											
7. Что такое кривая функционирования приемника (ROC, receiver operating characteristic) и кривая компромисса между ошибками детектирования (DET, detection error tradeoff)?

**8. Список литературы**

1. Hsu H.P. Schaum’s Ouline of Analog and Digital Communications. McGraw-Hill, 2003 ([ссылка](http://www.oasitech.it/upl/ese_schaum.pdf)).

2. Hsu H.P. Schaum’s Ouline of Theory and Problems of Probability, Random Variables, and Random Processes. McGraw-Hill, 1997 ([ссылка](http://webpages.iust.ac.ir/matashbar/teaching/schaum_probability.pdf)).